# VL-JEPA: Vision-Language Joint Embedding Predictive Architecture

## Overview
This notebook implements VL-JEPA, aligning video embeddings from V-JEPA2 with text embeddings using a contrastive learning approach.

**Architecture:**
- **Context Encoder**: V-JEPA2 ViT-L (frozen, 304M params) - processes video into visual embeddings
- **Predictor**: Llama-3.2-1B layers 8-15 (trainable, 490M params) - transforms visual to shared space
- **Target Encoder**: Gemma-2-2B (trainable, 2B params) - produces text embeddings
- **Loss**: InfoNCE contrastive loss in 2048-dim shared embedding space

In [ ]:
# Cell 1: Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    VJEPA2Model,
    LlamaForCausalLM
)
from datasets import load_dataset
from tqdm import tqdm
import numpy as np

device = torch.device('cuda:0')
print(f'Using device: {device}')

In [ ]:
# Cell 2: Load V-JEPA2 Vision Encoder (frozen)
print('Loading V-JEPA2 ViT-L on GPU 0...')
vjepa = VJEPA2Model.from_pretrained(
    'facebook/vjepa2-vit-large-16x256x256',
    torch_dtype=torch.float16
).to(device)

# Freeze all parameters - V-JEPA2 stays frozen
vjepa.eval()
for param in vjepa.parameters():
    param.requires_grad = False

vision_dim = vjepa.config.hidden_size  # 1024
print(f'V-JEPA2 loaded: {vision_dim}-dim embeddings')

In [ ]:
# Cell 3: Load Llama-3.2-1B Predictor
print('\nLoading Llama-3.2-1B...')
llama = LlamaForCausalLM.from_pretrained(
    'meta-llama/Llama-3.2-1B',
    torch_dtype=torch.float16
).to(device)

# Use full model but freeze first 8 layers (0-7)
# Layers 8-15 will be trainable
predictor = llama.model  # Full LlamaModel
predictor_dim = llama.config.hidden_size  # 2048

# Freeze embedding and first 8 layers
for param in predictor.embed_tokens.parameters():
    param.requires_grad = False
for i in range(8):
    for param in predictor.layers[i].parameters():
        param.requires_grad = False

print(f'Llama predictor loaded: {predictor_dim}-dim, layers 8-15 trainable')

In [ ]:
# Cell 4: Load Gemma-2-2B Text Encoder
print('\nLoading Gemma-2-2B...')
text_model = AutoModel.from_pretrained(
    'google/gemma-2-2b',
    torch_dtype=torch.float16
).to(device)
text_dim = text_model.config.hidden_size  # 2304

tokenizer = AutoTokenizer.from_pretrained('google/gemma-2-2b')
tokenizer.pad_token = tokenizer.eos_token

print(f'Gemma text encoder loaded: {text_dim}-dim')

In [ ]:
# Cell 5: Dataset
class MSRVTTDataset(Dataset):
    def __init__(self, split='train'):
        self.dataset = load_dataset('AlexZigma/msr-vtt', split=split)
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        item = self.dataset[idx]
        
        # Video: [T, H, W, C] -> [T, C, H, W] with 16 frames, 256x256
        frames = item['video'][:16]  # Take first 16 frames
        if len(frames) < 16:
            frames = frames + [frames[-1]] * (16 - len(frames))
        
        frames = torch.stack([torch.from_numpy(f).permute(2, 0, 1) for f in frames])
        frames = F.interpolate(frames, size=(256, 256), mode='bilinear', align_corners=False)
        frames = frames.float() / 255.0
        
        # Text
        caption = item['caption']
        tokens = tokenizer(
            caption,
            max_length=77,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'pixels': frames,
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0)
        }

train_dataset = MSRVTTDataset('train')
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
print(f'Dataset loaded: {len(train_dataset)} samples')

In [ ]:
# Cell 6: VL-JEPA Model with All Fixes
class VLJEPA(nn.Module):
    def __init__(self, vision, predictor, text, embed_dim=2048):
        super().__init__()
        self.vision = vision  # V-JEPA2 on GPU
        self.predictor = predictor  # Full Llama model
        self.text = text  # Gemma
        
        # FIX: Proper initialization with xavier_uniform and .half()
        self.proj_in = nn.Linear(vision_dim, predictor_dim)
        nn.init.xavier_uniform_(self.proj_in.weight)
        nn.init.zeros_(self.proj_in.bias)
        self.proj_in = self.proj_in.to(device).half()
        
        self.pred_proj = nn.Linear(predictor_dim, embed_dim)
        nn.init.xavier_uniform_(self.pred_proj.weight)
        nn.init.zeros_(self.pred_proj.bias)
        self.pred_proj = self.pred_proj.to(device).half()
        
        self.text_proj = nn.Linear(text_dim, embed_dim)
        nn.init.xavier_uniform_(self.text_proj.weight)
        nn.init.zeros_(self.text_proj.bias)
        self.text_proj = self.text_proj.to(device).half()
        
        self.temperature = nn.Parameter(torch.tensor(0.07, dtype=torch.float16).to(device))
    
    def forward(self, pixels, input_ids, attention_mask):
        # Handle device transfers internally
        pixels = pixels.to(device)
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        
        # Vision encoder (frozen)
        with torch.no_grad():
            visual = self.vision(pixels).last_hidden_state.float()
        
        # CRITICAL FIX: Use single combined operation to avoid NaN corruption
        # DO NOT use .to(device).half() - causes NaN on multi-GPU setups
        visual = visual.to(device, dtype=torch.float16)
        
        # Project to predictor dim
        h = self.proj_in(visual)
        
        # Predictor (full Llama model handles position embeddings internally)
        predictor_out = self.predictor(
            inputs_embeds=h,
            use_cache=False,
            return_dict=True
        )
        h = predictor_out.last_hidden_state
        
        # Project predictor output
        pred = self.pred_proj(h)
        pred = F.normalize(pred, dim=-1)
        
        # Text encoder
        text_out = self.text(input_ids, attention_mask=attention_mask)
        target = text_out.last_hidden_state
        
        # Project text output
        target = self.text_proj(target)
        target = F.normalize(target, dim=-1)
        
        return pred, target
    
    def compute_loss(self, pred, target, mask):
        # FIX: Ensure mask is on same device as pred/target
        mask = mask.to(pred.device)
        
        # Pool sequences
        pred_pool = pred.mean(dim=1)
        target_pool = (target * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp(min=1)
        
        # Normalize
        pred_pool = F.normalize(pred_pool, dim=-1)
        target_pool = F.normalize(target_pool, dim=-1)
        
        # InfoNCE loss
        temp = self.temperature.clamp(min=0.01, max=1.0)
        sim = torch.matmul(pred_pool, target_pool.T) / temp
        
        labels = torch.arange(sim.shape[0], device=sim.device)
        loss_v2t = F.cross_entropy(sim, labels)
        loss_t2v = F.cross_entropy(sim.T, labels)
        
        return (loss_v2t + loss_t2v) / 2

# Create model
model = VLJEPA(vjepa, predictor, text_model)
print('\n✓ VL-JEPA model created with all fixes')

In [ ]:
# Cell 7: Optimizer
trainable = list(model.predictor.layers[8:16].parameters()) + \
            list(model.text.parameters()) + \
            list(model.proj_in.parameters()) + \
            list(model.pred_proj.parameters()) + \
            list(model.text_proj.parameters()) + \
            [model.temperature]

optimizer = torch.optim.AdamW(trainable, lr=1e-4)
print(f'Optimizer created: {len(trainable)} parameter groups')

In [ ]:
# Cell 8: Training Loop
num_steps = 5
print(f'Training for {num_steps} steps...\n')

model.train()
model.vision.eval()  # Keep V-JEPA2 frozen in eval mode

for step, batch in enumerate(tqdm(train_loader, total=num_steps)):
    if step >= num_steps:
        break
    
    pixels = batch['pixels']
    input_ids = batch['input_ids']
    attention_mask = batch['attention_mask']
    
    # Forward (model handles GPU transfers)
    pred, target = model(pixels, input_ids, attention_mask)
    loss = model.compute_loss(pred, target, attention_mask)
    
    # Backward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    print(f'Step {step+1} | Loss: {loss.item():.4f} | Temp: {model.temperature.item():.4f}')

print('\n✓ Training complete!')